# Microsoft Retriever 1 (preview)

**Microsoft Retriever 1** (`microsoft-retriever-1`) is a retrieval model **hosted inside Foundry IQ**. It plans queries on the service side, so you don't have to connect, authenticate or pay for an external chat model just to plan retrieval.

This notebook uses the space exploration index from [prepare-dataset](prepare-dataset.ipynb) and shows Retriever 1 in three ways:

1. **Direct agentic search**: `query_type=QueryType.AGENTIC` on an ordinary index query, no knowledge base needed
2. **A knowledge base that uses only Retriever 1**, with `minimal` reasoning effort
3. **`auto` reasoning effort**: Retriever 1 handles what it can and escalates to your external model (for example `gpt-5.4-mini`) only when a request needs deeper planning

It ends with a side-by-side comparison of latency, planners and evidence.

![Three ways to use Microsoft Retriever 1: direct agentic search, a Retriever 1-only knowledge base at minimal effort, and auto effort that escalates to an external model only when needed](../docs/images/retriever-1.svg)

## Prerequisites

| Requirement | Notes |
|---|---|
| [prepare-dataset.ipynb](prepare-dataset.ipynb) completed | Provides the `space-exploration` index |
| Search service on the **Standard Knowledge Retrieval** plan | Required for Retriever 1. See [Enable or disable agentic retrieval billing](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-enable-disable) |
| Search service in a [region that supports Microsoft Retriever 1](https://learn.microsoft.com/azure/search/search-region-support#microsoft-retriever-1): **Japan East, North Europe, West Central US or West US 3** (as of November 2026) | Elsewhere, agentic queries fail with *"Agentic retrieval is not available in this region"* |
| Roles: **Search Service Contributor**, **Search Index Data Reader** | Create the knowledge source and knowledge bases, and query |
| An LLM deployment (`LLM_DEPLOYMENT`) | Only for the `auto` section; the search identity needs **Cognitive Services User** on the Foundry resource |

> **Billing:** Retriever 1 is billed on **input tokens** processed during query planning (output tokens are free). This is separate from Foundry IQ retrieval tokens and from Azure OpenAI charges. See [Use Microsoft Retriever 1](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-use-microsoft-retriever).

## Setup

In [ ]:
# %pip install azure-search-documents==12.2.0b1 azure-identity python-dotenv

In [ ]:
import json
import os
import time

from azure.core.credentials import AzureKeyCredential
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeBaseMicrosoftRetriever1Model,
    KnowledgeSourceReference,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    KnowledgeBaseRetrievalRequest,
    KnowledgeRetrievalAutoReasoningEffort,
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalMediumReasoningEffort,
    KnowledgeRetrievalMinimalReasoningEffort,
    KnowledgeRetrievalOutputMode,
    KnowledgeRetrievalSemanticIntent,
    SearchIndexKnowledgeSourceParams,
)
from azure.search.documents.models import AgenticSearchOptions, QueryDebugMode, QueryType, VectorizableTextQuery
from dotenv import load_dotenv

load_dotenv(override=True)  # pick up .env edits without restarting the kernel

search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
search_api_key = os.environ.get("AZURE_SEARCH_API_KEY")
search_credential = AzureKeyCredential(search_api_key) if search_api_key else DefaultAzureCredential()

ai_services_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"].split("/api/projects")[0]
llm_deployment = os.environ["LLM_DEPLOYMENT"]
llm_model = os.environ["LLM_MODEL"]

INDEX_NAME = "space-exploration"
KNOWLEDGE_SOURCE = "space-mr1-ks"
KB_RETRIEVER_ONLY = "space-mr1-kb"
KB_AUTO = "space-mr1-auto-kb"

index_client = SearchIndexClient(search_endpoint, search_credential)
search_client = SearchClient(search_endpoint, INDEX_NAME, search_credential)

print(f"Search endpoint:           {search_endpoint}")
print(f"External model for 'auto': {llm_deployment} / {llm_model}")

## Step 1: Direct agentic search

The simplest way to use Retriever 1 is to set the query type to `agentic` on a normal index query. You pass the **full natural-language question**. Retriever 1 decides which searches to run against the index, and the debug info shows:

- `termination_reason`: `modelStopped` (finished normally), `maxRuntimeExceeded`, or `modelFailed`
- `sub_queries`: the searches it issued
- `usage`: planning and reasoning tokens

We compare it against a plain hybrid query for the same multi-part question.

### Helpers

In [ ]:
def hybrid_search(question: str, top: int = 5) -> list[dict]:
    results = search_client.search(
        search_text=question,
        vector_queries=[VectorizableTextQuery(text=question, k_nearest_neighbors=50, fields="text_vector")],
        query_type=QueryType.SEMANTIC,
        select=["title", "chunk"],
        top=top,
    )
    return list(results)


def agentic_search(question: str, top: int = 5, filter_expr: str | None = None):
    """Return (documents, agentic debug info) for a Retriever 1 agentic query."""
    results = search_client.search(
        search_text=question,
        query_type=QueryType.AGENTIC,
        agentic_options=AgenticSearchOptions(max_runtime_in_seconds=60),
        filter=filter_expr,
        select=["title", "chunk"],
        top=top,
        debug=QueryDebugMode.AGENTIC,
    )
    documents = list(results)
    debug = results.get_debug_info()
    return documents, (debug.agentic if debug else None)


def print_documents(documents: list[dict]) -> None:
    for doc in documents:
        print(f"  {doc['title']:<28} {' '.join(doc['chunk'].split())[:100]}…")


def print_agentic_debug(agentic) -> None:
    if agentic is None:
        print("No agentic debug info returned")
        return
    reason = getattr(agentic.termination_reason, "value", agentic.termination_reason)
    print(f"Stopped because:  {reason}")
    if agentic.usage:
        print(f"Tokens used:      planning {agentic.usage.agentic_planning_tokens:,}, reasoning {agentic.usage.reasoning_tokens:,}")
    for sub_query in agentic.sub_queries or []:
        print(f"  subquery (iteration {sub_query.iteration}): {sub_query.search}")


### Run it

In [ ]:
QUESTION = "Who performed the first spacewalk, on which mission, and which cosmodrome did that mission launch from?"

print("=== Plain hybrid + semantic search ===")
print_documents(hybrid_search(QUESTION))

In [ ]:
start = time.perf_counter()
documents, agentic = agentic_search(QUESTION)
print(f"=== Agentic search with Microsoft Retriever 1 ({time.perf_counter() - start:.1f}s) ===")
print_documents(documents)
print()
print_agentic_debug(agentic)

Filters still apply, and they constrain all of the evidence Retriever 1 is allowed to use. Here we restrict it to the Soviet programme articles.

> Direct agentic search doesn't support `skip`, `order_by`, `include_total_count`, facets, highlighting, query rewrites, speller, semantic answers or captions.

In [ ]:
soviet_titles = "Voskhod 2,Alexei Leonov,Baikonur Cosmodrome,Vostok 1,Yuri Gagarin,Sergei Korolev"
documents, agentic = agentic_search(QUESTION, filter_expr=f"search.in(title, '{soviet_titles}', ',')")
print(f"Articles used (all inside the filter): {', '.join(sorted({doc['title'] for doc in documents}))}")
print_agentic_debug(agentic)


## Step 2: A knowledge base that uses only Retriever 1

Knowledge bases add citations, an activity log, multi-source support and an MCP endpoint for agents. To use Retriever 1 in a knowledge base:

- Set `models` to `[KnowledgeBaseMicrosoftRetriever1Model()]`
- Use **`minimal`** reasoning effort. A Retriever 1-only knowledge base doesn't support `low` or `medium`, or knowledge-source selection
- Send **exactly one semantic intent** with the complete question. Multiple intents skip Retriever 1 and return `206` with a warning

### Create the knowledge source and knowledge base

In [ ]:
knowledge_source = SearchIndexKnowledgeSource(
    name=KNOWLEDGE_SOURCE,
    description="Wikipedia chunks about space exploration: missions, astronauts, agencies, launch sites and spacecraft.",
    search_index_parameters=SearchIndexKnowledgeSourceParameters(
        search_index_name=INDEX_NAME,
        source_data_fields=[SearchIndexFieldReference(name=f) for f in ("title", "source_url", "chunk")],
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source)
print(f"Knowledge source '{KNOWLEDGE_SOURCE}' created or updated")

In [ ]:
kb_retriever_only = KnowledgeBase(
    name=KB_RETRIEVER_ONLY,
    description="Space exploration knowledge base planned entirely by Microsoft Retriever 1.",
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE)],
    models=[KnowledgeBaseMicrosoftRetriever1Model()],
    retrieval_reasoning_effort=KnowledgeRetrievalMinimalReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)
index_client.create_or_update_knowledge_base(kb_retriever_only)
print(f"Knowledge base '{KB_RETRIEVER_ONLY}' created or updated")

### Retrieval helpers

`retrieve()` sends one semantic intent and optionally overrides the reasoning effort for that request. The other helpers read the **activity log** (which models planned the request, tokens, timing) and the **references** (which articles supplied evidence).

In [ ]:
EFFORTS = {
    "minimal": KnowledgeRetrievalMinimalReasoningEffort,
    "low": KnowledgeRetrievalLowReasoningEffort,
    "medium": KnowledgeRetrievalMediumReasoningEffort,
    "auto": KnowledgeRetrievalAutoReasoningEffort,
}


def retrieve(kb_name: str, question: str, effort: str | None = None):
    """Return (response, elapsed seconds). `effort` overrides the KB default for this request only."""
    client = KnowledgeBaseRetrievalClient(search_endpoint, search_credential, knowledge_base_name=kb_name)
    request = KnowledgeBaseRetrievalRequest(
        intents=[KnowledgeRetrievalSemanticIntent(search=question)],
        retrieval_reasoning_effort=EFFORTS[effort]() if effort else None,
        output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
        include_activity=True,
        knowledge_source_params=[SearchIndexKnowledgeSourceParams(
            knowledge_source_name=KNOWLEDGE_SOURCE,
            include_references=True,
            include_reference_source_data=True,
        )],
    )
    start = time.perf_counter()
    response = client.retrieve(request)
    return response, time.perf_counter() - start


def planning_models(response) -> list[str]:
    return [
        record.model.model_name
        for record in response.activity or []
        if record.type == "modelQueryPlanning" and getattr(record, "model", None)
    ]


def reference_titles(response) -> set[str]:
    return {(ref.source_data or {}).get("title") for ref in response.references or []} - {None}


def show_retrieval(response, elapsed: float) -> None:
    print(f"Elapsed: {elapsed:.1f}s\n")
    print(f"  {'step':<22} {'model':<22} {'tokens in/out':<16} time")
    for record in response.activity or []:
        model = getattr(getattr(record, "model", None), "model_name", "") or "-"
        tokens = f"{record.input_tokens:,}/{record.output_tokens:,}" if getattr(record, "input_tokens", None) is not None else "-"
        elapsed_ms = f"{record.elapsed_ms} ms" if record.elapsed_ms else "-"
        print(f"  {record.type:<22} {model:<22} {tokens:<16} {elapsed_ms}")
        if record.error:
            print(f"    error: {record.error}")
    titles = sorted(reference_titles(response))
    print(f"\nEvidence from {len(titles)} articles: {', '.join(titles)}")


### Run it

In [ ]:
response, elapsed = retrieve(KB_RETRIEVER_ONLY, QUESTION)
show_retrieval(response, elapsed)

Look for a `modelQueryPlanning` record with model `microsoft-retriever-1`: that confirms Retriever 1 planned the request. If it's missing, check the prerequisites (plan, region) and that you sent a single intent.

With `extractiveData` output, the knowledge base returns the grounding chunks themselves (no generated answer). This is what an agent or your own LLM call would receive:

In [ ]:
grounding = json.loads(response.response[0].content[0].text)
print(f"{len(grounding)} grounding chunks returned. First 3:\n")
for item in grounding[:3]:
    print(f"[ref_id {item['ref_id']}] {item['title']}")
    print(f"  {' '.join(item['content'].split())[:250]}…\n")


## Step 3: `auto` reasoning effort

Requests vary in difficulty. With `auto`, the knowledge base picks the depth per request:

1. **Initial pass** (minimal): a fast lookup. If the top result is relevant enough, it returns immediately
2. **Retriever 1** runs deeper retrieval if the first pass isn't enough
3. **External model, medium effort**: Retriever 1 hands planning to your model when the request needs it

`auto` **requires an external model** on the knowledge base, so this knowledge base lists both models. Requests with more than one intent, or knowledge bases with sources that don't support the initial pass (for example MCP or web), start at medium instead.

In [ ]:
kb_auto = KnowledgeBase(
    name=KB_AUTO,
    description="Space exploration knowledge base with auto reasoning effort.",
    knowledge_sources=[KnowledgeSourceReference(name=KNOWLEDGE_SOURCE)],
    models=[
        KnowledgeBaseMicrosoftRetriever1Model(),
        KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
            resource_url=ai_services_endpoint, deployment_name=llm_deployment, model_name=llm_model,
        )),
    ],
    retrieval_reasoning_effort=KnowledgeRetrievalAutoReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.EXTRACTIVE_DATA,
)
index_client.create_or_update_knowledge_base(kb_auto)
print(f"Knowledge base '{KB_AUTO}' created or updated")

### Run an easy and a hard question

In [ ]:
EASY = "When did Sputnik 1 launch?"

response, elapsed = retrieve(KB_AUTO, EASY)
show_retrieval(response, elapsed)
print(f"Planning models used: {', '.join(planning_models(response)) or 'none (the initial pass was enough)'}")


In [ ]:
HARD = (
    "Compare how the Soviet, American and Chinese programmes each put their first person into orbit: "
    "who flew, on which spacecraft, in what year, and from which launch site?"
)

response, elapsed = retrieve(KB_AUTO, HARD)
show_retrieval(response, elapsed)
print(f"Planning models used: {', '.join(planning_models(response)) or 'none (the initial pass was enough)'}")


For the easy question you should see little or no model planning: the initial pass was sufficient. For the hard question you'll typically see `microsoft-retriever-1`, and possibly a second `modelQueryPlanning` record for your external model, which means Retriever 1 escalated.

## Step 4: Side-by-side comparison

The same questions through four paths:

| Path | Planner |
|---|---|
| Hybrid search | None (one query) |
| KB, `minimal`, Retriever 1 only | Microsoft Retriever 1 |
| KB, `auto` | Retriever 1, escalating to the external model |
| KB, `medium` (request override on the auto KB) | External model |

### Helpers

In [ ]:
PATHS = [
    ("mr1-minimal", KB_RETRIEVER_ONLY, None),
    ("auto", KB_AUTO, None),
    ("medium", KB_AUTO, "medium"),
]


def compare_paths(question: str) -> list[tuple]:
    start = time.perf_counter()
    hybrid_titles = {doc["title"] for doc in hybrid_search(question)}
    rows = [(question, "hybrid", time.perf_counter() - start, len(hybrid_titles), "-")]
    for label, kb, effort in PATHS:
        response, elapsed = retrieve(kb, question, effort)
        rows.append((question, label, elapsed, len(reference_titles(response)), ", ".join(planning_models(response)) or "-"))
    return rows


def print_comparison(rows: list[tuple]) -> None:
    print(f"{'question':<42} {'path':<12} {'secs':>6} {'articles':>8}  planners")
    for question, path, secs, n_titles, planners in rows:
        print(f"{question[:40]:<42} {path:<12} {secs:>6.1f} {n_titles:>8}  {planners}")

### Run it

In [ ]:
QUESTIONS = [
    EASY,
    QUESTION,
    HARD,
    "Which space telescopes observe in infrared, and where in space is each one positioned?",
]

rows = [row for question in QUESTIONS for row in compare_paths(question)]
print_comparison(rows)

**Reading the table**

- *articles*: how many distinct Wikipedia articles contributed evidence. Multi-part questions need several (for example astronaut, mission **and** launch site)
- *planners*: which models planned the request. `auto` should use the external model only on the hardest questions
- Retriever 1 avoids most external-model calls while still breaking multi-part questions into several searches. Compare its latency against `medium`

## Troubleshooting

| Symptom | Fix |
|---|---|
| No `microsoft-retriever-1` planning record | Confirm the Standard Knowledge Retrieval plan and a supported region; send exactly one semantic intent |
| `206` with an `agenticReasoning` warning | Multiple intents were sent, so Retriever 1 was skipped |
| `modelQueryPlanning` record contains an `error` | Retriever 1 started but didn't finish. With `minimal`, retained results are returned as `206`; with `auto`, planning is handed to the external model |
| `low`/`medium` rejected on the Retriever 1-only KB | Expected: add an external model (as in the `auto` KB) to use them |

## Clean up

Deletes the knowledge bases and knowledge source created here. The `space-exploration` index is kept for the other notebooks.

In [ ]:
# index_client.delete_knowledge_base(KB_AUTO)
# index_client.delete_knowledge_base(KB_RETRIEVER_ONLY)
# index_client.delete_knowledge_source(KNOWLEDGE_SOURCE)
# print("Deleted knowledge bases and knowledge source")

## Summary

- **Direct agentic search** (`QueryType.AGENTIC`) gives Retriever 1 planning on any index, with filters, and no knowledge base
- A **Retriever 1-only knowledge base** (`minimal`) gives citations and an activity log without connecting an external planning model
- **`auto`** combines a fast initial pass, Retriever 1 and your external model, and only pays for deeper planning when a request needs it

### Next steps

- [topic-graph.ipynb](topic-graph.ipynb): broad questions about themes across the corpus
- [entity-graph.ipynb](entity-graph.ipynb): multi-hop questions that follow relationships between entities